In [5]:
from io import StringIO
import json
import os

import boto3
import pandas as pd
pd.set_option("display.max_columns",30)


In [6]:
aws_access_key=os.getenv("AWS_ACCESS_KEY")
aws_secret_key=os.getenv("AWS_SECRET_KEY")


In [7]:
s3=boto3.client("s3",aws_access_key_id=aws_access_key, aws_secret_access_key=aws_secret_key)

In [8]:
bucket="cubix-chichago-taxi-bb-v2"
dim_community_areas_path="transformed_data/dim_community_areas/dim_community_areas.csv"
dim_company_path="transformed_data/dim_company/dim_company.csv"
dim_date_path="transformed_data/dim_date/dim_date.csv"
dim_payment_type_path="transformed_data/dim_payment_type/dim_payment_type.csv"
dim_weather_path="transformed_data/dim_weather/"
fact_taxi_trips_path="transformed_data/fact_taxi_trips/"


In [9]:
def read_file_from_s3(s3, bucket:str, key: str, file_format:str ="csv"):
    #---    
    #Reads a csv or json file from an S3 bucket

    #:param s3:         S3 cliebt
    #:param bucket:     name of the S3 bucket where the file is stored
    #:param key:        Path within the S3 bucket
    #:file_format:      json or csv
    #---
    response = s3.get_object(Bucket=bucket, Key=key)
    content = response['Body'].read().decode('utf-8')

    if file_format == 'csv':
        return pd.read_csv(StringIO(content))
    elif file_format == 'json':
        return json.loads(content)
    else:
        raise ValueError("Unsupported file format. Use 'csv' or 'json'.")        


In [10]:
dim_community_areas=read_file_from_s3(s3, bucket, dim_community_areas_path)
dim_company=read_file_from_s3(s3, bucket, dim_company_path)
dim_date=read_file_from_s3(s3, bucket, dim_date_path)
dim_payment_type=read_file_from_s3(s3, bucket, dim_payment_type_path)



In [11]:
fact_taxi_trips_list=[]
dim_weather_list=[]


In [12]:
for file in s3.list_objects(Bucket=bucket, Prefix=fact_taxi_trips_path)["Contents"]:
    taxi_key=file["Key"]
    taxi_raw_filename=file["Key"].split("/")[-1]

    if taxi_raw_filename.split(".")[-1] == "csv":
        daily_file_name=fact_taxi_trips_path + taxi_raw_filename
        taxi_trips_daily=read_file_from_s3(s3, bucket, daily_file_name)

        fact_taxi_trips_list.append(taxi_trips_daily)
        print(f"{taxi_raw_filename} has been added")

         
#         taxi_raw_content = read_file_from_s3(s3, bucket, taxi_key)


     


taxi_2025-10-11.csv has been added
taxi_2025-10-12.csv has been added
taxi_2025-10-13.csv has been added
taxi_2025-10-14.csv has been added


In [13]:
fact_taxi_trips=pd.concat(fact_taxi_trips_list, ignore_index=True)


In [14]:
fact_taxi_trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68976 entries, 0 to 68975
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   trip_id                     68976 non-null  object 
 1   taxi_id                     68976 non-null  object 
 2   trip_start_timestamp        68976 non-null  object 
 3   trip_end_timestamp          68976 non-null  object 
 4   trip_seconds                68976 non-null  int64  
 5   trip_miles                  68976 non-null  float64
 6   pickup_community_area_id    68976 non-null  int64  
 7   dropoff_community_area_id   68976 non-null  int64  
 8   fare                        68976 non-null  float64
 9   tips                        68976 non-null  float64
 10  tolls                       68976 non-null  float64
 11  extras                      68976 non-null  float64
 12  trip_total                  68976 non-null  float64
 13  pickup_centroid_latitude    689

In [15]:
for file in s3.list_objects(Bucket=bucket, Prefix=dim_weather_path)["Contents"]:
    weather_key=file["Key"]

    weather_raw_filename=file["Key"].split("/")[-1]
    if weather_raw_filename.split(".")[-1] == "csv":
        daily_file_name=dim_weather_path + weather_raw_filename
        weather_daily = read_file_from_s3(s3, bucket, daily_file_name)

        dim_weather_list.append(weather_daily)
        print(f"{weather_raw_filename} has been added")



weather_2025-10-11.csv has been added
weather_2025-10-12.csv has been added
weather_2025-10-13.csv has been added
weather_2025-10-14.csv has been added


In [16]:
dim_weather=pd.concat(dim_weather_list, ignore_index=True)

In [17]:
dim_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   datetime       96 non-null     object 
 1   temperature    96 non-null     float64
 2   wind_speed     96 non-null     float64
 3   rain           96 non-null     float64
 4   precipitation  96 non-null     float64
dtypes: float64(4), object(1)
memory usage: 3.9+ KB


### Create datamodell

In [18]:
fact_taxi_trips_full=pd.merge(fact_taxi_trips, dim_weather, left_on="datetime_for_weather", right_on="datetime")
fact_taxi_trips_full=fact_taxi_trips_full.drop(columns=["datetime_for_weather","datetime"])

In [19]:
fact_taxi_trips_full=pd.merge(fact_taxi_trips_full, dim_company, left_on="company_id", right_on="company_id")
fact_taxi_trips_full=fact_taxi_trips_full.drop(columns=["company_id"])

In [20]:
fact_taxi_trips_full=pd.merge(fact_taxi_trips_full, dim_payment_type, left_on="payment_type_id", right_on="payment_type_id")
fact_taxi_trips_full=fact_taxi_trips_full.drop(columns=["payment_type_id"])

In [21]:
fact_taxi_trips_full=pd.merge(fact_taxi_trips_full, dim_community_areas, left_on="dropoff_community_area_id", right_on="area_code")
fact_taxi_trips_full=fact_taxi_trips_full.drop(columns=["dropoff_community_area_id", "area_code"])
fact_taxi_trips_full.rename(columns={"community name":"dropoff_community_area_name"},inplace=True)


In [22]:
fact_taxi_trips_full=pd.merge(fact_taxi_trips_full, dim_community_areas, left_on="pickup_community_area_id", right_on="area_code")
fact_taxi_trips_full=fact_taxi_trips_full.drop(columns=["pickup_community_area_id", "area_code"])
fact_taxi_trips_full.rename(columns={"community name":"pickup_community_area_name"},inplace=True)


In [23]:
dim_date["date"] = pd.to_datetime(dim_date["date"])
fact_taxi_trips_full["trip_start_timestamp"] = pd.to_datetime(fact_taxi_trips_full["trip_start_timestamp"])

fact_taxi_trips_full["trip_start_date"]=pd.to_datetime(fact_taxi_trips_full["trip_start_timestamp"].dt.date)


In [24]:
fact_taxi_trips_full=pd.merge(fact_taxi_trips_full, dim_date, left_on="trip_start_date", right_on="date")
fact_taxi_trips_full=fact_taxi_trips_full.drop(columns=["trip_start_date","date"])


In [25]:
fact_taxi_trips_full.head(1)

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,...,:created_at,:updated_at,temperature,wind_speed,rain,precipitation,company,payment_type,dropoff_community_area_name,pickup_community_area_name,year,month,day,day_of_week,is_weekend
0,e05851cbf4fa2744e9158ee4a0b5fefa7aeaaa88,f73d5459dcae455d9d7eb464fb7689b48cf9518d3cf1b7...,2025-10-11 23:45:00,2025-10-12T00:00:00.000,367,2.27,8.5,0.0,0.0,20.0,29.0,41.980264,-87.913625,41.980264,-87.913625,...,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,15.4,12.8,0.0,0.0,City Service,Credit Card,O'Hare[11],O'Hare[11],2025,10,11,6,True


In [26]:
fact_taxi_trips_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68976 entries, 0 to 68975
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   trip_id                      68976 non-null  object        
 1   taxi_id                      68976 non-null  object        
 2   trip_start_timestamp         68976 non-null  datetime64[ns]
 3   trip_end_timestamp           68976 non-null  object        
 4   trip_seconds                 68976 non-null  int64         
 5   trip_miles                   68976 non-null  float64       
 6   fare                         68976 non-null  float64       
 7   tips                         68976 non-null  float64       
 8   tolls                        68976 non-null  float64       
 9   extras                       68976 non-null  float64       
 10  trip_total                   68976 non-null  float64       
 11  pickup_centroid_latitude     68976 non-nu